# PRISM - 01 Model Training
Predictive Maintenance on AI4I 2020 dataset. Goal ROC-AUC >=0.88 (README.md).

**Exploration only:** the reproducible source of truth is now `src/prism/ml/`. Use the documented CLI to train, evaluate, persist artifacts, and test README payloads.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
import joblib
# optional: XGBoost if available
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('xgboost not installed -> fallback RandomForest (pip install xgboost to enable)')
print('HAS_XGB:', HAS_XGB)

## 2. Load Data

In [ ]:
df = pd.read_csv('ai4i2020.csv')
print(df.shape)
df.head()

## 3. Drop Non-Predictive ID Columns

**Why drop `UDI` and `Product ID`:**
- `UDI` = row index 1..10000, unique identifier, no physics meaning
- `Product ID` = like L47181/M14860, high-cardinality ID encoding Type + serial, leaky & no generalization
- Both cause overfit, fake importance, data leakage if model memorizes IDs
- Keep only maths features: Type, Air/Process temp, RPM, Torque, Tool wear (README.md)
- RNF column also dropped later (random failure not predictable, but keep for EDA)

In [ ]:
# --- USER REQUEST: delete UDI and Product ID ---
cols_to_drop = ['UDI', 'Product ID']
print('Before:', df.columns.tolist())
df = df.drop(columns=cols_to_drop)
print('After:', df.columns.tolist())
print('Shape:', df.shape)
# verify drop stuck
assert 'UDI' not in df.columns and 'Product ID' not in df.columns, 'drop failed!'
df.head()

In [ ]:
# save cleaned copy for downstream pipeline (optional)
df.to_csv('ai4i2020_clean.csv', index=False)
print('saved ai4i2020_clean.csv with', df.shape)

## 4. EDA - Imbalance + Distributions

In [ ]:
print(df['Machine failure'].value_counts())
print(df['Machine failure'].value_counts(normalize=True))
print(df[['TWF','HDF','PWF','OSF','RNF']].sum())
print(df['Type'].value_counts())
df.describe()

## 5. Feature Engineering
- temp_diff = Process - Air (heat dissipation signal HDF)
- power = Torque * RPM (power failure / overstrain signal)
- Keep Type categorical for one-hot

In [ ]:
df['temp_diff'] = df['Process temperature [K]'] - df['Air temperature [K]']
df['power'] = df['Torque [Nm]'] * df['Rotational speed [rpm]']
# features for binary failure
feature_cols = ['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'temp_diff', 'power']
X = df[feature_cols]
y = df['Machine failure']
print(X.head())
print('X shape', X.shape, 'y mean', y.mean())

## 6. Preprocess + Split (stratified)

In [ ]:
numeric_features = ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'temp_diff', 'power']
categorical_features = ['Type']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train {X_train.shape} fail={y_train.sum()} ({y_train.mean():.3%})')
print(f'Test  {X_test.shape} fail={y_test.sum()} ({y_test.mean():.3%})')

## 7. Train + Eval (ROC-AUC >=0.88)
Imbalance: class_weight balanced / scale_pos_weight

In [ ]:
def build_model():
    if HAS_XGB:
        # scale_pos_weight = neg/pos
        spw = (len(y_train) - y_train.sum()) / y_train.sum()
        clf = XGBClassifier(scale_pos_weight=spw, eval_metric='logloss', random_state=42, n_estimators=300, max_depth=6, learning_rate=0.05)
        print(f'Using XGB scale_pos_weight={spw:.1f}')
    else:
        clf = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1)
        print('Using RandomForest balanced')
    return clf

clf = build_model()
pipe = Pipeline([('prep', preprocessor), ('clf', clf)])
pipe.fit(X_train, y_train)

y_prob = pipe.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, y_prob)
print(f'ROC-AUC: {auc:.4f} -> {"PASS >=0.88" if auc>=0.88 else "FAIL"}')
print(classification_report(y_test, (y_prob>0.5).astype(int)))
print(confusion_matrix(y_test, (y_prob>0.5).astype(int)))

## 8. Threshold Tuning + Cross-Val

In [ ]:
from sklearn.metrics import precision_recall_curve
thresholds = [0.3,0.5,0.7,0.8]
for t in thresholds:
    preds = (y_prob>t).astype(int)
    print(f't={t} auc already {auc:.3f}')

cv = StratifiedKFold(5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc')
print(f'CV ROC-AUC {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}', cv_scores)

## 9. Persist Production Artifact

In [ ]:
# Artifact persistence is centralized in the reproducible package.
# Run from the project root after completing this exploration:
print(r'.\.venv\Scripts\python.exe -m prism.ml.train --config configs/training.json')

## 10. Test README.md Payloads (Section 4)
Payload 1 HEALTHY, Payload 2 CRITICAL

In [ ]:
import pandas as pd
def predict_payload(metrics):
    # metrics keys: air_temperature_k, process_temperature_k, rotational_speed_rpm, torque_nm, tool_wear_min
    row = {
        'Type': 'M',  # default, README payloads no Type given -> assume M
        'Air temperature [K]': metrics['air_temperature_k'],
        'Process temperature [K]': metrics['process_temperature_k'],
        'Rotational speed [rpm]': metrics['rotational_speed_rpm'],
        'Torque [Nm]': metrics['torque_nm'],
        'Tool wear [min]': metrics['tool_wear_min'],
    }
    row['temp_diff'] = row['Process temperature [K]'] - row['Air temperature [K]']
    row['power'] = row['Torque [Nm]'] * row['Rotational speed [rpm]']
    Xq = pd.DataFrame([row])[feature_cols]
    prob = pipe.predict_proba(Xq)[0,1]
    status = 'HEALTHY' if prob < 0.7 else 'CRITICAL_WARNING'
    return prob, status

p1 = {"air_temperature_k": 298.1, "process_temperature_k": 308.6, "rotational_speed_rpm": 1551, "torque_nm": 42.8, "tool_wear_min": 21}
p2 = {"air_temperature_k": 304.5, "process_temperature_k": 313.2, "rotational_speed_rpm": 1201, "torque_nm": 71.2, "tool_wear_min": 215}

prob1, status1 = predict_payload(p1)
prob2, status2 = predict_payload(p2)
print(f'Payload1 prob={prob1:.4f} status={status1} -> expect HEALTHY ~0.02')
print(f'Payload2 prob={prob2:.4f} status={status2} -> expect CRITICAL_WARNING')
assert status1=='HEALTHY', 'payload1 should be HEALTHY'
assert status2=='CRITICAL_WARNING', 'payload2 should be CRITICAL'
print('payload test pass')